## API Call

In [12]:
import requests
import pandas as pd


def get_nsf_awards(full_name, rpp=25):

    full_name = str(full_name).strip()

    print(f"Searching NSF: {full_name}")

    def search_awards(search_parameter):

        awards_all = []
        offset = 0

        while True:

            params = {
                search_parameter: full_name,
                "rpp": rpp,
                "offset": offset
            }

            response = requests.get(
                "https://api.nsf.gov/services/v1/awards.json",
                params=params,
                timeout=30
            )

            response.raise_for_status()

            awards = response.json()["response"].get("award", [])

            if not awards:
                break

            awards_all.extend(awards)

            if len(awards) < rpp:
                break

            offset += rpp

        return awards_all

    # Search PI and Co-PI awards
    pi_awards = search_awards("pdPIName")
    copi_awards = search_awards("coPDPI")

    # Add role
    for award in pi_awards:
        award["Role"] = "PI"

    for award in copi_awards:
        award["Role"] = "Co-PI"

    # Combine
    df = pd.DataFrame(pi_awards + copi_awards)

    # Add searched person's information
    df["Full Name"] = full_name

    # Desired output columns
    output_columns = [
        "Full Name",
        "Participants",
        "Role",
        "Title",
        "Abstract",
        "Award Type",
        "Journal",
        "Amount",
        "NSF Email",
        "Institute",
        "Start Date",
        "End Date"
    ]

    # NSF field names
    columns = {
        "title": "Title",
        "abstractText": "Abstract",
        "transType": "Award Type",
        "jrnl": "Journal",
        "estimatedTotalAmt": "Amount",
        "piEmail": "NSF Email",
        "awardee": "Institute",
        "startDate": "Start Date",
        "expDate": "End Date"
    }

    df = df.rename(columns=columns)

    # Ensure every desired column exists
    for col in output_columns:
        if col not in df.columns:
            df[col] = pd.NA

    return df[output_columns]

In [13]:
df = pd.read_excel("mhti_accepted.xlsx")[["name"]]

print(df)

                 name
0       Alicia Bargar
1    Andrew D. Miller
2      Audrey Boruvka
3             Bin Xie
4            Dan Ding
..                ...
338     Natasha Chaku
339       Phoebe Tran
340    Tyler Prochnow
341       Xueqing Liu
342        Yiwen Dong

[343 rows x 1 columns]


In [14]:
import time

def get_nsf_awards_for_people(people, delay=0.2):

    all_results = []

    for _, person in people.iterrows():

        full_name = person["name"]

        try:

            result = get_nsf_awards(
                full_name=full_name
            )

            if not result.empty:
                all_results.append(result)

                print(
                    f"  Found {len(result)} matching award(s)"
                )

            else:
                print("  No matching awards found")

        except requests.exceptions.RequestException as e:

            print(
                f"  API error for "
                f"{full_name}: {e}"
            )

        except Exception as e:

            print(
                f"  Error for "
                f"{full_name}: {e}"
            )

        time.sleep(delay)

    if not all_results:
        return pd.DataFrame(
            columns=[
                "Title",
                "Abstract",
                "Award Type",
                "Amount",
                "Email",
                "Institute",
                "Date"
            ]
        )

    return pd.concat(
        all_results,
        ignore_index=True
    )

In [15]:
results = get_nsf_awards_for_people(df)

Searching NSF: Alicia Bargar
  No matching awards found
Searching NSF: Andrew D. Miller
  Found 2 matching award(s)
Searching NSF: Audrey Boruvka
  No matching awards found
Searching NSF: Bin Xie
  No matching awards found
Searching NSF: Dan Ding
  Found 6 matching award(s)
Searching NSF: Diana Ramos
  No matching awards found
Searching NSF: Eun Kyoung Choe
  Found 5 matching award(s)
Searching NSF: François Modave
  No matching awards found
Searching NSF: Ingo Vetter
  No matching awards found
Searching NSF: Ju Gao
  No matching awards found
Searching NSF: Judy Hahn
  No matching awards found
Searching NSF: Kathleen (Katie) Reilly
  No matching awards found
Searching NSF: Mahbubur Rahman
  No matching awards found
Searching NSF: Ming Ji
  Found 1 matching award(s)
Searching NSF: Quyen Epstein-Ngo
  No matching awards found
Searching NSF: Ting Zhu
  Found 22 matching award(s)
Searching NSF: Bo-Jhang Ho
  No matching awards found
Searching NSF: David Do
  No matching awards found
Search

In [16]:
results["Role"].unique()

array(['PI', 'Co-PI'], dtype=object)

In [17]:
results

,Full Name,Participants,Role,Title,Abstract,Award Type,Journal,Amount,NSF Email,Institute,Start Date,End Date
0,Andrew D. Miller,NaN,PI,Career: Family Resilience Technologies: Augmen...,This research will explore how social technolo...,Continuing Grant,"[{'artTitl': 'Helping Their Child, Helping Eac...",520935,andrewm@iupui.edu,TRUSTEES OF INDIANA UNIVERSITY,06/01/2021,12/31/2026
1,Andrew D. Miller,NaN,PI,CRII:SCH: Parent-2-Parent: Supporting Dyadic C...,"When a child is hospitalized, existing support...",Standard Grant,[{'artTitl': 'Family Care Coordination in the ...,174791,andrewm@iupui.edu,TRUSTEES OF INDIANA UNIVERSITY,06/01/2019,05/31/2022
2,Dan Ding,NaN,PI,Quality of Life Technology Research Experience...,Technical Summary\r\n\r\nStudents participatin...,Standard Grant,[{'artTitl': 'Helping Hands: Requirements for ...,345739,dad5@pitt.edu,UNIVERSITY OF PITTSBURGH - OF THE COMMONWEALTH...,04/01/2014,03/31/2018
3,Dan Ding,NaN,PI,The Quality of Life Technology Model for Gradu...,The Quality of Life Technology Enrichment (QuO...,Standard Grant,[{'artTitl': 'Pipeline for Students with Disab...,199849,dad5@pitt.edu,UNIVERSITY OF PITTSBURGH - OF THE COMMONWEALTH...,01/01/2012,12/31/2014
4,Dan Ding,NaN,PI,Research Experience for Undergraduates (REU) i...,This three year renewal REU Site program at th...,Standard Grant,[{'artTitl': 'Retention Strategies for Student...,356700,dad5@pitt.edu,UNIVERSITY OF PITTSBURGH - OF THE COMMONWEALTH...,05/01/2011,04/30/2014
...,...,...,...,...,...,...,...,...,...,...,...,...
222,Yanmin Gong,NaN,PI,CRII: NeTS: Embracing Dynamic Spectrum Sharing...,The proliferation of wireless devices and band...,Standard Grant,[{'artTitl': 'Geo-Indistinguishablility for Cr...,175000,yanmin.gong@tamu.edu,THE UNIVERSITY OF TEXAS AT SAN ANTONIO,10/01/2019,09/30/2023
223,Yanmin Gong,NaN,Co-PI,Collaborative Research: CNS Core: Medium: Towa...,Recent emerging federated learning (FL) allows...,Continuing Grant,[{'artTitl': 'Hybrid Quantum-Classical Benders...,250000,yuanxiong.guo@utsa.edu,THE UNIVERSITY OF TEXAS AT SAN ANTONIO,10/01/2021,09/30/2027
224,Yanmin Gong,NaN,Co-PI,RAPID: Collaborative: Location Privacy Preserv...,The recent pandemic of COVID-19 has caused a p...,Standard Grant,[{'artTitl': 'Geo-Indistinguishablility for Cr...,100000,yuanxiong.guo@utsa.edu,THE UNIVERSITY OF TEXAS AT SAN ANTONIO,05/01/2020,04/30/2022
225,Ziping Xu,NaN,PI,Collaborative Research: FDT-BioTech: Mechanist...,Parkinson’s disease affects more than one mill...,Standard Grant,NaN,400000,zipingxu@unc.edu,UNIVERSITY OF NORTH CAROLINA AT CHAPEL HILL,09/15/2026,08/31/2029


In [23]:
results.to_excel("NSF_raw2.xlsx", index=False)

## Merge with Demographic Data

In [26]:
import pandas as pd

df_nsf = pd.read_excel("NSF_raw2.xlsx")
df_demo = pd.read_excel("mhti_accepted.xlsx")
df_demo.rename(columns={'name': 'Full Name'}, inplace=True)

In [27]:
df_nsf2 = df_nsf.copy()
df_demo2 = df_demo.copy()

# --- Merge onto NSF data ---

df_nsf2 = df_nsf2.merge(
    df_demo,
    on=["Full Name"],
    how="left"
)

df_nsf2

,Full Name,Participants,Role,Title,Abstract,Award Type,Journal,Amount,NSF Email,Institute,Start Date,End Date,cohort,affiliation,openalex_author_id,google_scholar_url,google_scholar_user_id,note
0,Andrew D. Miller,NaN,PI,Career: Family Resilience Technologies: Augmen...,This research will explore how social technolo...,Continuing Grant,"[{'artTitl': ""Helping Their Child, Helping Eac...",520935,andrewm@iupui.edu,TRUSTEES OF INDIANA UNIVERSITY,06/01/2021,12/31/2026,2015,University of Washington,A5027248126,https://scholar.google.com/citations?user=U5Qe...,U5QeuUcAAAAJ,NaN
1,Andrew D. Miller,NaN,PI,CRII:SCH: Parent-2-Parent: Supporting Dyadic C...,"When a child is hospitalized, existing support...",Standard Grant,[{'artTitl': 'Family Care Coordination in the ...,174791,andrewm@iupui.edu,TRUSTEES OF INDIANA UNIVERSITY,06/01/2019,05/31/2022,2015,University of Washington,A5027248126,https://scholar.google.com/citations?user=U5Qe...,U5QeuUcAAAAJ,NaN
2,Dan Ding,NaN,PI,Quality of Life Technology Research Experience...,Technical Summary\n\n\n\nStudents participatin...,Standard Grant,[{'artTitl': 'Helping Hands: Requirements for ...,345739,dad5@pitt.edu,UNIVERSITY OF PITTSBURGH - OF THE COMMONWEALTH...,04/01/2014,03/31/2018,2015,University of Pittsburgh,NaN,https://scholar.google.com/citations?user=WzE5...,WzE5dbYAAAAJ,NaN
3,Dan Ding,NaN,PI,The Quality of Life Technology Model for Gradu...,The Quality of Life Technology Enrichment (QuO...,Standard Grant,[{'artTitl': 'Pipeline for Students with Disab...,199849,dad5@pitt.edu,UNIVERSITY OF PITTSBURGH - OF THE COMMONWEALTH...,01/01/2012,12/31/2014,2015,University of Pittsburgh,NaN,https://scholar.google.com/citations?user=WzE5...,WzE5dbYAAAAJ,NaN
4,Dan Ding,NaN,PI,Research Experience for Undergraduates (REU) i...,This three year renewal REU Site program at th...,Standard Grant,[{'artTitl': 'Retention Strategies for Student...,356700,dad5@pitt.edu,UNIVERSITY OF PITTSBURGH - OF THE COMMONWEALTH...,05/01/2011,04/30/2014,2015,University of Pittsburgh,NaN,https://scholar.google.com/citations?user=WzE5...,WzE5dbYAAAAJ,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
222,Yanmin Gong,NaN,PI,CRII: NeTS: Embracing Dynamic Spectrum Sharing...,The proliferation of wireless devices and band...,Standard Grant,[{'artTitl': 'Geo-Indistinguishablility for Cr...,175000,yanmin.gong@tamu.edu,THE UNIVERSITY OF TEXAS AT SAN ANTONIO,10/01/2019,09/30/2023,2025,University of Texas at San Antonio,A5035287134,https://scholar.google.com/citations?user=o3BG...,o3BGIEMAAAAJ,NaN
223,Yanmin Gong,NaN,Co-PI,Collaborative Research: CNS Core: Medium: Towa...,Recent emerging federated learning (FL) allows...,Continuing Grant,"[{'artTitl': ""Hybrid Quantum-Classical Benders...",250000,yuanxiong.guo@utsa.edu,THE UNIVERSITY OF TEXAS AT SAN ANTONIO,10/01/2021,09/30/2027,2025,University of Texas at San Antonio,A5035287134,https://scholar.google.com/citations?user=o3BG...,o3BGIEMAAAAJ,NaN
224,Yanmin Gong,NaN,Co-PI,RAPID: Collaborative: Location Privacy Preserv...,The recent pandemic of COVID-19 has caused a p...,Standard Grant,[{'artTitl': 'Geo-Indistinguishablility for Cr...,100000,yuanxiong.guo@utsa.edu,THE UNIVERSITY OF TEXAS AT SAN ANTONIO,05/01/2020,04/30/2022,2025,University of Texas at San Antonio,A5035287134,https://scholar.google.com/citations?user=o3BG...,o3BGIEMAAAAJ,NaN
225,Ziping Xu,NaN,PI,Collaborative Research: FDT-BioTech: Mechanist...,Parkinson’s disease affects more than one mill...,Standard Grant,NaN,400000,zipingxu@unc.edu,UNIVERSITY OF NORTH CAROLINA AT CHAPEL HILL,09/15/2026,08/31/2029,2025,Harvard University,A5007965098,https://scholar.google.com/citations?user=V-Vc...,V-VcaYIAAAAJ,NaN


In [29]:
df_nsf2.to_excel("NSF_demographic2.xlsx", index=False)